In [13]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import com.microsoft.spark.fabric

print("Starting Silver → Gold Transformation...")



# Configuration
SILVER_TABLE = "lh_CustomerChurn.silver.customerchurn"
GOLD_TABLE = "gold.customer_summary"
BATCH_SIZE = 1000

  # Step 1: Read silver data (incremental)
df_silver = spark.sql(f"""SELECT * FROM {SILVER_TABLE} """)
display(df_silver)

StatementMeta(, d216fc22-707e-4e59-a7c7-610077286868, 15, Finished, Available, Finished, False)

Starting Silver → Gold Transformation...


SynapseWidget(Synapse.DataFrame, ba8dba1b-9305-4cd5-854a-2339b69180ac)

In [14]:
df_silver.count()

StatementMeta(, d216fc22-707e-4e59-a7c7-610077286868, 16, Finished, Available, Finished, False)

25025

In [16]:
df_silver = df_silver.drop_duplicates()
df_silver.count()

StatementMeta(, d216fc22-707e-4e59-a7c7-610077286868, 18, Finished, Available, Finished, False)

5005

In [17]:
row_count_before = df_silver.count()
print(f"Processing {row_count_before} records from silver layer...")
    
if row_count_before == 0:
    print("⚠ No new data to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

StatementMeta(, d216fc22-707e-4e59-a7c7-610077286868, 19, Finished, Available, Finished, False)

Processing 5005 records from silver layer...


In [19]:
# Step 2: Aggregate data

df_gold = (df_silver
    .orderBy(
        col("cr8b0_internetservicetype")
    )
    .groupBy(
        col("cr8b0_internetservicetype"),
        col("cr8b0_paymentmethod")
    )
    .agg(
        count("*").alias("transaction_count"),
        sum(col("cr8b0_totalcharges")).alias("TotalSumCharges")
    )
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

row_count_gold = df_gold.count()
print(f"Aggregated to {row_count_gold} gold records")


# Step 3: Write to gold (append, partitioned by load_date)
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold.CustomerChurn")
df_silver.write.mode("overwrite").synapsesql("dw_CustomerChurn.silver.CustomerChurn")

display(df_gold)

StatementMeta(, d216fc22-707e-4e59-a7c7-610077286868, 21, Finished, Available, Finished, False)

Aggregated to 12 gold records


SynapseWidget(Synapse.DataFrame, 7720b0ae-ea8d-4291-a3d6-a6c030b56e96)